# Deney 1 Frame — Real/Fake Dengeli Ortak Frame Seti

Bu notebook, daha önce videolardan çıkarılmış `real_frames` ve `fake_frames` görüntülerinden yüz içeren ortak bir veri alt kümesi oluşturur.

- **Genel toplam 3000 frame** seçilir.
- **1500 Real + 1500 Fake** olacak şekilde sınıflar dengelenir.
- Her sınıf kendi içinde **%80 train, %10 validation, %10 test** olarak ayrılır:
  - Real: 1200 train, 150 val, 150 test
  - Fake: 1200 train, 150 val, 150 test
- Kaynak klasörlerdeki mevcut `train/val/test` ayrımı korunur. Böylece aynı video veya aynı videoya ait benzer frame'lerin farklı veri bölümlerine karışması önlenir.
- Bir frame seçilmeden önce MediaPipe ile en az bir yüz içerip içermediği kontrol edilir.
- Seçilen bütün frame'ler metadata CSV dosyasına, yüz bulunamayanlar ise log dosyasına yazılır.
- Çıktı klasörleri işlem başlamadan önce temizlenir; eski çalıştırmadan kalan fazla görüntüler sonuçları bozmaz.

## 1. Google Drive Bağlantısı

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
!pip uninstall -y mediapipe
!pip install -q mediapipe==0.10.21

Found existing installation: mediapipe 1.0.0
Uninstalling mediapipe-1.0.0:
  Successfully uninstalled mediapipe-1.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 104.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 118.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.2/81.2 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 12.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have pro

## 2. Gerekli Kütüphaneler

In [2]:
!pip install mediapipe -q

import os
import cv2
import random
import shutil
import csv
import mediapipe as mp
from tqdm.notebook import tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.5/36.5 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 12.6 MB/s eta 0:00:00


## 3. Konfigürasyon

Kaynak klasör yapısı:

```text
AISC DeepFake Çalışması/
└── FaceForensics_Projesi/
    ├── real_frames/
    │   ├── train/
    │   ├── val/
    │   └── test/
    └── fake_frames/
        ├── train/
        ├── val/
        └── test/
```

Oluşturulacak çıktı:

```text
AISC DeepFake Çalışması/
└── Deney 1 Frame/
    ├── Real/
    │   ├── train/  (1200)
    │   ├── val/    (150)
    │   └── test/   (150)
    └── Fake/
        ├── train/  (1200)
        ├── val/    (150)
        └── test/   (150)
```

In [4]:
# ============================================================
# DRIVE YOLLARI
# ============================================================

REAL_FRAMES_ROOT = "/content/drive/MyDrive/AISC DeepFake Çalışmaları/FaceForensics_Projesi/real_frames"
FAKE_FRAMES_ROOT = "/content/drive/MyDrive/AISC DeepFake Çalışmaları/FaceForensics_Projesi/fake_frames"

OUTPUT_ROOT = "/content/drive/MyDrive/AISC DeepFake Çalışması/Deney 1 Frame"

# GENEL TOPLAM = 3000
TOTAL_TARGET = 3000
TARGET_PER_CLASS = TOTAL_TARGET // 2  # 1500 Real + 1500 Fake

TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
TEST_RATIO  = 0.10

# Her sınıf için kesin hedefler
TARGETS_PER_SPLIT = {
    "train": int(TARGET_PER_CLASS * TRAIN_RATIO),  # 1200
    "val":   int(TARGET_PER_CLASS * VAL_RATIO),    # 150
    "test":  int(TARGET_PER_CLASS * TEST_RATIO),   # 150
}

FACE_MIN_CONFIDENCE = 0.50
RANDOM_SEED = 42
IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

NO_FACE_LOG_PATH  = os.path.join(OUTPUT_ROOT, "yuzsuz_frameler.txt")
METADATA_CSV_PATH = os.path.join(OUTPUT_ROOT, "secim_metadata.csv")

assert TOTAL_TARGET % 2 == 0, "Toplam hedef sayı Real ve Fake arasında eşit bölünebilmelidir."
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9
assert sum(TARGETS_PER_SPLIT.values()) == TARGET_PER_CLASS

# Kaynak klasörleri kontrol et
for root in [REAL_FRAMES_ROOT, FAKE_FRAMES_ROOT]:
    if not os.path.isdir(root):
        raise FileNotFoundError(f"Kaynak klasör bulunamadı: {root}")
    for split in ["train", "val", "test"]:
        split_path = os.path.join(root, split)
        if not os.path.isdir(split_path):
            raise FileNotFoundError(f"Kaynak split klasörü bulunamadı: {split_path}")

# Eski sonuçların yeni sayımları bozmaması için yalnızca çıktı sınıf klasörlerini temizle
os.makedirs(OUTPUT_ROOT, exist_ok=True)

for sinif in ["Real", "Fake"]:
    class_dir = os.path.join(OUTPUT_ROOT, sinif)
    if os.path.exists(class_dir):
        shutil.rmtree(class_dir)
    for split in ["train", "val", "test"]:
        os.makedirs(os.path.join(class_dir, split), exist_ok=True)

print("Çıktı klasörleri temizlendi ve yeniden oluşturuldu.")
print("Genel hedef:", TOTAL_TARGET)
print("Sınıf başına hedef:", TARGET_PER_CLASS)
print("Split hedefleri:", TARGETS_PER_SPLIT)

Çıktı klasörleri temizlendi ve yeniden oluşturuldu.
Genel hedef: 3000
Sınıf başına hedef: 1500
Split hedefleri: {'train': 1200, 'val': 150, 'test': 150}


## 4. Yardımcı Fonksiyonlar

In [5]:
def klasordeki_tum_frameleri_bul(root_dir):
    """Verilen klasörün altındaki bütün görüntü dosyalarını sıralı şekilde döndürür."""
    paths = []
    for dirpath, _, filenames in os.walk(root_dir):
        for filename in filenames:
            if filename.lower().endswith(IMG_EXTENSIONS):
                paths.append(os.path.join(dirpath, filename))
    return sorted(paths)


def yuz_var_mi(image_path, detector):
    """Görüntü okunabiliyorsa ve en az bir yüz bulunuyorsa True döndürür."""
    image = cv2.imread(image_path)
    if image is None:
        return False

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    result = detector.process(image_rgb)
    return bool(result.detections)


def yuzlu_frameleri_sec(frame_pool, target_count, detector, log_rows, sinif_adi, split_adi, seed):
    """Belirli bir sınıf/split havuzundan tam hedef kadar yüzlü frame seçer."""
    candidates = frame_pool.copy()
    random.Random(seed).shuffle(candidates)

    selected = []

    for path in tqdm(candidates, desc=f"{sinif_adi}/{split_adi} yüz kontrolü"):
        if len(selected) == target_count:
            break

        if yuz_var_mi(path, detector):
            selected.append(path)
        else:
            log_rows.append({
                "sinif": sinif_adi,
                "split": split_adi,
                "dosya": path,
            })

    if len(selected) < target_count:
        raise RuntimeError(
            f"{sinif_adi}/{split_adi} için yeterli yüzlü frame bulunamadı. "
            f"Hedef={target_count}, bulunan={len(selected)}, havuz={len(frame_pool)}"
        )

    return selected


def benzersiz_hedef_adi(sinif_adi, split_adi, index, src_path):
    """Kaynak isimleri çakışsa bile benzersiz ve düzenli çıktı adı üretir."""
    extension = os.path.splitext(src_path)[1].lower()
    return f"{sinif_adi.lower()}_{split_adi}_{index:05d}{extension}"


def frameleri_kopyala(selected, sinif_adi, split_adi, metadata_rows):
    """Seçilen frame'leri hedef klasöre kopyalar ve metadata ekler."""
    dest_dir = os.path.join(OUTPUT_ROOT, sinif_adi, split_adi)

    for index, src_path in enumerate(selected):
        filename = benzersiz_hedef_adi(sinif_adi, split_adi, index, src_path)
        dst_path = os.path.join(dest_dir, filename)
        shutil.copy2(src_path, dst_path)

        metadata_rows.append({
            "sinif": sinif_adi,
            "split": split_adi,
            "orijinal_yol": src_path,
            "yeni_yol": dst_path,
        })


def goruntu_sayisi(folder):
    return sum(
        1 for name in os.listdir(folder)
        if name.lower().endswith(IMG_EXTENSIONS)
    )

## 5. Kaynak Split Havuzlarının Toplanması

In [6]:
kaynak_havuzlar = {
    "Real": {
        split: klasordeki_tum_frameleri_bul(os.path.join(REAL_FRAMES_ROOT, split))
        for split in ["train", "val", "test"]
    },
    "Fake": {
        split: klasordeki_tum_frameleri_bul(os.path.join(FAKE_FRAMES_ROOT, split))
        for split in ["train", "val", "test"]
    },
}

for sinif_adi in ["Real", "Fake"]:
    print(f"\n{sinif_adi} kaynak sayıları")
    for split_adi in ["train", "val", "test"]:
        count = len(kaynak_havuzlar[sinif_adi][split_adi])
        target = TARGETS_PER_SPLIT[split_adi]
        print(f"  {split_adi}: {count} görüntü | hedef: {target}")
        if count < target:
            raise RuntimeError(
                f"{sinif_adi}/{split_adi} klasöründe hedef sayıdan daha az görüntü var."
            )


Real kaynak sayıları
  train: 21705 görüntü | hedef: 1200
  val: 2745 görüntü | hedef: 150
  test: 2742 görüntü | hedef: 150

Fake kaynak sayıları
  train: 135550 görüntü | hedef: 1200
  val: 16903 görüntü | hedef: 150
  test: 16932 görüntü | hedef: 150


## 6. Yüz Kontrolüyle Dengeli Frame Seçimi

In [7]:
yuzsuz_log_rows = []
secilenler = {"Real": {}, "Fake": {}}

mp_face_detection = mp.solutions.face_detection

# FaceForensics++ videolarındaki yüzler genellikle yakın/orta mesafede olduğu için model_selection=0
with mp_face_detection.FaceDetection(
    model_selection=0,
    min_detection_confidence=FACE_MIN_CONFIDENCE
) as detector:

    for class_index, sinif_adi in enumerate(["Real", "Fake"]):
        for split_index, split_adi in enumerate(["train", "val", "test"]):
            seed = RANDOM_SEED + class_index * 100 + split_index

            secilenler[sinif_adi][split_adi] = yuzlu_frameleri_sec(
                frame_pool=kaynak_havuzlar[sinif_adi][split_adi],
                target_count=TARGETS_PER_SPLIT[split_adi],
                detector=detector,
                log_rows=yuzsuz_log_rows,
                sinif_adi=sinif_adi,
                split_adi=split_adi,
                seed=seed,
            )

for sinif_adi in ["Real", "Fake"]:
    print(f"\n{sinif_adi} seçilen sayıları")
    for split_adi in ["train", "val", "test"]:
        print(f"  {split_adi}: {len(secilenler[sinif_adi][split_adi])}")

print("\nYüz bulunmadığı veya görüntü okunamadığı için atlanan:", len(yuzsuz_log_rows))

AttributeError: module 'mediapipe' has no attribute 'solutions'

## 7. Seçilen Frame'lerin Yeni Klasörlere Kopyalanması

In [ ]:
metadata_rows = []

for sinif_adi in ["Real", "Fake"]:
    for split_adi in ["train", "val", "test"]:
        frameleri_kopyala(
            selected=secilenler[sinif_adi][split_adi],
            sinif_adi=sinif_adi,
            split_adi=split_adi,
            metadata_rows=metadata_rows,
        )

print("Toplam kopyalanan frame:", len(metadata_rows))

## 8. Log ve Metadata Dosyalarının Kaydedilmesi

In [ ]:
with open(NO_FACE_LOG_PATH, "w", encoding="utf-8") as f:
    for row in yuzsuz_log_rows:
        f.write(f"{row['sinif']}\t{row['split']}\t{row['dosya']}\n")

with open(METADATA_CSV_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["sinif", "split", "orijinal_yol", "yeni_yol"]
    )
    writer.writeheader()
    writer.writerows(metadata_rows)

print("Yüzsüz frame logu:", NO_FACE_LOG_PATH)
print("Seçim metadata dosyası:", METADATA_CSV_PATH)

## 9. Sonuçların Kesin Doğrulanması

In [ ]:
expected_total = 0
actual_total = 0

print("=" * 65)
print("ÇIKTI DOĞRULAMA")
print("=" * 65)

for sinif_adi in ["Real", "Fake"]:
    class_total = 0

    for split_adi in ["train", "val", "test"]:
        folder = os.path.join(OUTPUT_ROOT, sinif_adi, split_adi)
        actual = goruntu_sayisi(folder)
        expected = TARGETS_PER_SPLIT[split_adi]

        print(f"{sinif_adi:4s}/{split_adi:5s}: {actual:4d} | beklenen: {expected:4d}")
        assert actual == expected, (
            f"{sinif_adi}/{split_adi} sayısı yanlış: {actual}, beklenen: {expected}"
        )

        class_total += actual
        expected_total += expected
        actual_total += actual

    assert class_total == TARGET_PER_CLASS
    print(f"{sinif_adi} toplam: {class_total}\n")

assert actual_total == TOTAL_TARGET
assert len(metadata_rows) == TOTAL_TARGET
assert len({row['orijinal_yol'] for row in metadata_rows}) == TOTAL_TARGET

print(f"GENEL TOPLAM: {actual_total}")
print("Dağılım doğru: 1500 Real + 1500 Fake")
print("Her sınıfta dağılım doğru: 1200 train + 150 val + 150 test")

## 10. Özet

In [ ]:
print("=" * 65)
print("DENEY 1 FRAME SEÇİMİ TAMAMLANDI")
print("=" * 65)

for sinif_adi in ["Real", "Fake"]:
    counts = {
        split: len(secilenler[sinif_adi][split])
        for split in ["train", "val", "test"]
    }
    print(
        f"{sinif_adi}: {sum(counts.values())} "
        f"(train={counts['train']}, val={counts['val']}, test={counts['test']})"
    )

print(f"Toplam frame: {len(metadata_rows)}")
print(f"Yüzsüz/okunamayan frame: {len(yuzsuz_log_rows)}")
print(f"Çıktı klasörü: {OUTPUT_ROOT}")